# Planner Agent

LangGraph-based agent that breaks a travel request into actionable tasks (airfare, hotel, car rental).

In [ ]:
import asyncio
import logging
from collections.abc import AsyncIterable
from typing import Any, Literal

import uvicorn
from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field

import prompts
from common import BaseAgent, TaskList, build_a2a_app, init_api_key

load_dotenv()
init_api_key()
logger = logging.getLogger(__name__)

In [ ]:
class ResponseFormat(BaseModel):
    """Respond to the user in this format."""
    status: Literal['input_required', 'completed', 'error'] = 'input_required'
    question: str = Field(description='Input needed from the user')
    content: TaskList = Field(description='List of tasks when the plan is generated')


class LangGraphPlannerAgent(BaseAgent):
    def __init__(self):
        super().__init__(agent_name='PlannerAgent', description='Breakdown user request into tasks', content_types=['text', 'text/plain'])
        self.model = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.0)
        self.graph = create_react_agent(
            self.model, checkpointer=MemorySaver(),
            prompt=prompts.PLANNER_COT_INSTRUCTIONS,
            response_format=ResponseFormat, tools=[],
        )

    async def stream(self, query, session_id, task_id) -> AsyncIterable[dict[str, Any]]:
        config = {'configurable': {'thread_id': session_id}}
        for item in self.graph.stream({'messages': [('user', query)]}, config, stream_mode='values'):
            message = item['messages'][-1]
            if isinstance(message, AIMessage):
                yield {'response_type': 'text', 'is_task_complete': False, 'require_user_input': False, 'content': message.content}
        yield self._get_response(config)

    def _get_response(self, config):
        resp = self.graph.get_state(config).values.get('structured_response')
        if isinstance(resp, ResponseFormat):
            if resp.status == 'completed':
                return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': resp.content.model_dump()}
            return {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': resp.question}
        return {'is_task_complete': False, 'require_user_input': True, 'content': 'Unable to process request. Please try again.'}

In [ ]:
app = build_a2a_app(LangGraphPlannerAgent(), 'agent_cards/planner_agent.json')

config = uvicorn.Config(app=app, host='localhost', port=10102, log_level='info')
server = uvicorn.Server(config)
asyncio.ensure_future(server.serve())
print('Planner Agent running at http://localhost:10102')